
# NYC Taxi → product_ec  
## Notebook d’exploration & calibration (référence)

### Objectif
Ce notebook sert de support d’exploration et de calibration pour le détournement  
du dataset **NYC Taxi** vers une table métier `product_ec`.




In [1]:

from pathlib import Path
import pandas as pd
import numpy as np



## Chargement des données


In [4]:

DATA_PATH = Path("./data/yellow_tripdata_2025-01.parquet")
df = pd.read_parquet(DATA_PATH)
df.shape


(3475226, 20)


## Aperçu des données


In [7]:

df.columns
df.head()


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0



## Colonnes utilisées pour le détournement

- `PULocationID` → entrepôts  
- `DOLocationID` → produits  
- `fare_amount` → empreinte process  
- `trip_distance` → empreinte transport  
- `total_amount` → proxy de stock  

Les autres colonnes sont ignorées.



## Analyse statistique des colonnes clés


In [8]:

df[["fare_amount", "trip_distance"]].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
)


,fare_amount,trip_distance
count,3.475226e+06,3.475226e+06
mean,1.708180e+01,5.855126e+00
std,4.634729e+02,5.646016e+02
min,-9.000000e+02,0.000000e+00
50%,1.211000e+01,1.670000e+00
90%,3.498000e+01,7.700000e+00
95%,5.200000e+01,1.183000e+01
99%,7.230000e+01,1.950000e+01
max,8.633721e+05,2.764236e+05


In [9]:

(df["fare_amount"] <= 0).mean(), (df["trip_distance"] <= 0).mean()


(np.float64(0.04187238470246252), np.float64(0.026154558005723944))

In [10]:

df["fare_amount"].sort_values(ascending=False).head(10)


1780915    863372.12
1404958      2450.90
1832645      1309.20
2358836       950.00
2358837       950.00
2358832       950.00
1346041       936.80
2358828       900.00
562143        900.00
2353881       899.99
Name: fare_amount, dtype: float64

In [11]:

df["trip_distance"].sort_values(ascending=False).head(10)


3275762    276423.57
3188501    276099.95
3240337    222167.49
3112173    206137.99
3374854    202771.63
3081597    189687.43
3069039    181139.99
3312563    168079.57
3195037    167452.94
3138510    164959.95
Name: trip_distance, dtype: float64


## Nettoyage (clipping des outliers)


In [13]:

fare = df["fare_amount"].clip(
    lower=0,
    upper=df["fare_amount"].quantile(0.99)
)

dist = df["trip_distance"].clip(
    lower=0,
    upper=df["trip_distance"].quantile(0.99)
)



## Référentiel métier – Textile (rappel)

- Empreinte Process : 2 → 50 kg CO2eq  
- Empreinte Transport : 0.05 → 8 kg CO2eq  

Règle :
- hors transport aérien,  
  **ec_process ≈ 15–20 × ec_transport**



## Calibration de l’empreinte Process


In [14]:

ec_process = 2 + 48 * (
    (fare - fare.min()) / (fare.max() - fare.min())
)

ec_process.describe(percentiles=[0.5, 0.9, 0.95])


count    3.475226e+06
mean     1.331028e+01
std      9.927470e+00
min      2.000000e+00
50%      1.003983e+01
90%      2.522324e+01
95%      3.652282e+01
max      5.000000e+01
Name: fare_amount, dtype: float64


## Calibration de l’empreinte Transport


In [15]:

ec_transport = 0.05 + 7.95 * (
    (dist - dist.min()) / (dist.max() - dist.min())
)

ec_transport.describe(percentiles=[0.5, 0.9, 0.95])


count    3.475226e+06
mean     1.295939e+00
std      1.590551e+00
min      5.000000e-02
50%      7.308462e-01
90%      3.189231e+00
95%      4.873000e+00
max      8.000000e+00
Name: trip_distance, dtype: float64


## Vérification du ratio Process / Transport


In [16]:

(ec_process / ec_transport).describe(percentiles=[0.5, 0.9, 0.95])


count    3.475226e+06
mean     2.240711e+01
std      6.394035e+01
min      2.500000e-01
50%      1.367739e+01
90%      2.306972e+01
95%      3.014236e+01
max      1.000000e+03
dtype: float64


## Prototype minimal de détournement


In [17]:

N_PRODUCT_REF = 50_000
N_WAREHOUSE = 200

sample = df.sample(n=200_000, random_state=42).copy()

sample["id_warehouse"] = (sample["PULocationID"] % N_WAREHOUSE) + 1
sample["id_product_ref"] = (sample["DOLocationID"] % N_PRODUCT_REF) + 1

sample["ec_process"] = ec_process.loc[sample.index]
sample["ec_transport"] = ec_transport.loc[sample.index]
sample["ec_total"] = sample["ec_process"] + sample["ec_transport"]

sample["stock_qty"] = (
    (sample["total_amount"].clip(lower=0) /
     sample["total_amount"].quantile(0.95) * 199)
    .clip(1, 200)
    .round()
    .astype("int")
)

sample["id_product_ec"] = np.arange(1, len(sample) + 1)


In [18]:

sample[
    ["id_product_ec", "id_product_ref", "id_warehouse",
     "stock_qty", "ec_process", "ec_transport", "ec_total"]
].head(10)


,id_product_ec,id_product_ref,id_warehouse,stock_qty,ec_process,ec_transport,ec_total
1661588,1,167,39,36,7.709544,0.445462,8.155005
2269859,2,49,51,28,5.850622,0.294615,6.145238
1876274,3,238,37,45,8.174274,0.376154,8.550428
976447,4,69,32,55,9.103734,0.836846,9.940581
2962359,5,89,138,94,19.460581,1.860154,21.320735
2618157,6,264,141,42,7.244813,0.498462,7.743275
2800941,7,231,171,48,9.103734,0.592231,9.695965
2075873,8,44,44,43,9.568465,0.559615,10.128080
2052243,9,114,101,63,11.892116,0.743077,12.635193
1728339,10,142,164,43,9.568465,0.539231,10.107695
